In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    DIVISÃO TREINO / TESTE — SIN BRASIL                       ║
# ║        Split Temporal por Janela Móvel, Auditoria e Geração de Figuras       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : Universidade Federal do Ceará
#  Curso       : Engenharia Elétrica
#  Data        : Julho / 2026
#  Versão      : 3.0
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de divisão treino/teste da arquitetura de
#  dados do TCC. Ele consome o dataset consolidado do SIN Brasil (gerado nas
#  etapas anteriores de coleta e engenharia de dados) e o particiona em dois
#  subconjuntos temporais para uso em modelagem preditiva do PLD:
#
#    • TREINO : todo o histórico disponível, exceto a janela de teste
#    • TESTE  : os N_MESES_TESTE meses-calendário mais recentes do dataset
#               (janela móvel — recalculada a cada execução conforme os
#               dados mais atuais presentes no arquivo de entrada)
#
#  Diferente de um corte fixo por ano, a janela de teste "desliza" junto com
#  a atualização dos dados: conforme novos meses são coletados, os N meses
#  mais recentes sempre compõem o conjunto de teste, garantindo um cenário
#  de avaliação sempre atualizado e sem necessidade de ajuste manual de
#  constantes a cada nova execução.
#
#  Ao final da execução, o script gera:
#    (a) dois arquivos Parquet (treino e teste) prontos para modelagem;
#    (b) 6 figuras em alta resolução (300 dpi) para uso direto no TCC;
#    (c) 3 tabelas de auditoria exibidas no terminal (via Rich);
#    (d) 1 CSV de auditoria consolidada por ano e conjunto.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script não requer entradas manuais do usuário. Os parâmetros de controle
#  estão definidos nas constantes abaixo (seção "CONFIGURAÇÕES GLOBAIS"):
#
#    N_MESES_TESTE : quantidade de meses-calendário mais recentes reservados
#                    para teste  (padrão: 2)
#    BASE_PATH     : caminho raiz de armazenamento
#                    (/content/drive/MyDrive/TCC_PLD_Project)
#
#  Arquivo de entrada esperado:
#    • BASE_PATH / "2-DADOS" / "4-DADOS-UNIAO-GERAL" / "DADOS_BRASIL_WIDE.parquet"
#      → dataset consolidado do SIN Brasil, formato wide, contendo as colunas-
#        chave KEY_ANO, KEY_MES, KEY_DIA, KEY_HORA e as demais features numé-
#        ricas por submercado (sufixo _SUDESTE, _SUL, _NORDESTE, _NORTE).
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  1. Arquivos Parquet →  DIR_SAIDA / "treino" / TREINO_SIN_BRASIL.parquet
#                         DIR_SAIDA / "teste"  / TESTE_SIN_BRASIL.parquet
#     Compressão: snappy | Sem índice de linha | Mesmo schema do arquivo de entrada
#
#  2. Figuras PNG (300 dpi) → DIR_SAIDA / "figuras" /
#     • FIG01_proporcao_treino_teste.png   — pizza treino × teste
#     • FIG02_timeline_split.png           — linha do tempo mensal com corte
#     • FIG03_registros_por_ano.png        — registros por ano (empilhado)
#     • FIG04_nan_por_ano_split.png        — % de dados faltantes por ano
#     • FIG05_heatmap_nan_split.png        — heatmap NaN por região × ano
#     • FIG06_painel_sintetico.png         — painel 2×2 consolidado
#
#  3. CSV de auditoria → DIR_SAIDA / "figuras" / TAB_AUDITORIA_SPLIT.csv
#     Contém: Conjunto, Ano, Registros, Meses c/ Dados, NaN (%), Completude (%)
#
#  4. Saída no terminal (Rich):
#     • Tabela 1 — Resumo geral do split (registros, %, tamanho em MB)
#     • Tabela 2 — Auditoria temporal por ano (NaN, completude)
#     • Tabela 3 — Distribuição mensal de registros (treino vs teste)
#     • Painéis de progresso e relatório final
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ETAPAS DO PROCESSO (resumo)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  N°  Etapa                    Estratégia                    Descrição
#  01  Carregamento             read_parquet + dtype opt.     Lê o dataset Brasil e otimiza tipos
#  02  Identificação do corte   AAAAMM ordenado                Localiza os N meses mais recentes
#  03  Split                    Máscara booleana sem copy      Segrega treino/teste sem duplicar dados
#  04  Persistência             to_parquet (snappy)            Salva treino e teste em disco
#  05  Liberação de memória     del + gc.collect()              Libera df_raw após persistência
#  06  Auditoria                groupby leve por ano            Calcula NaN, completude, registros
#  07  Figuras                  matplotlib + seaborn            Gera 6 figuras para o TCC
#  08  Tabelas                  Rich Table                      Exibe relatórios no terminal
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Depende do arquivo consolidado gerado pela etapa de engenharia de dados
#     (script de transformação, v3.0). Certifique-se de que essa etapa foi
#     concluída com sucesso antes de executar este pipeline.
#
#  2. REGRA DE SPLIT — JANELA MÓVEL
#     O conjunto de TESTE corresponde sempre aos N_MESES_TESTE meses-calendário
#     mais recentes efetivamente presentes no dataset (coluna auxiliar AAAAMM,
#     construída apenas em memória a partir de KEY_ANO e KEY_MES). O restante
#     do histórico compõe o TREINO. Isso elimina a necessidade de atualizar
#     manualmente um ano de corte fixo a cada nova coleta de dados.
#
#  3. GERENCIAMENTO DE MEMÓRIA
#     O split é feito por máscara booleana diretamente sobre df_raw, sem
#     .copy(), e cada subset é persistido imediatamente em Parquet. Logo após
#     a gravação, df_raw e as máscaras são deletados e gc.collect() é chamado,
#     evitando picos de memória em ambientes com RAM limitada (Colab).
#
#  4. OTIMIZAÇÃO DE TIPOS
#     Colunas float64 são convertidas para float32 e colunas inteiras para o
#     menor dtype possível (int16/int32) antes do split, reduzindo o consumo
#     de memória em cerca de 50% nas colunas de dados.
#
#  5. AUDITORIA LEVE
#     A auditoria por ano lê apenas as colunas necessárias de cada arquivo já
#     persistido (não recarrega o dataset original), calculando NaN médio,
#     completude e número de meses com dados por (Conjunto, Ano).
#
#  6. FIGURAS COM GRANULARIDADE MENSAL
#     Como o corte pode cair no meio de um ano-calendário, a linha do tempo
#     (Figura 2) e o painel sintético (Figura 6) exibem os dados em resolução
#     mensal. As Figuras 3 e 4 exibem barras empilhadas/lado a lado por ano,
#     já que um mesmo ano pode conter meses de treino e de teste simultaneamente.
#
#  7. TRATAMENTO DE ERROS EM FIGURAS
#     Cada função de figura usa plt.rc_context + try/finally, garantindo que
#     a figura seja fechada (plt.close()) mesmo em caso de exceção durante a
#     geração, evitando vazamento de memória gráfica.
#
#  8. HEATMAP POR REGIÃO
#     A Figura 5 lê os arquivos Parquet já persistidos em blocos (colunas por
#     região), sem carregar o dataset inteiro em memória, para calcular o
#     percentual de NaN por região e por ano.
#
#  9. LIMITES CONHECIDOS
#     - Se o dataset tiver poucos meses (≤ N_MESES_TESTE), o split pode ficar
#       desbalanceado; o script emite um aviso nesse caso.
#     - A coluna auxiliar de período (AAAAMM) existe apenas em memória durante
#       a execução e não é persistida nos arquivos Parquet finais.
#
#  10. REPRODUTIBILIDADE
#      Os nomes dos arquivos de saída (TREINO_SIN_BRASIL.parquet e
#      TESTE_SIN_BRASIL.parquet) são fixos e determinísticos, sendo
#      sobrescritos a cada nova execução — a rastreabilidade do corte aplicado
#      fica registrada no CSV de auditoria e nas figuras geradas.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações numéricas auxiliares
#  pandas            1.5            Manipulação de DataFrames e Parquet
#  pyarrow           —              Engine de leitura/escrita Parquet
#  matplotlib        3.5            Geração das figuras estáticas
#  seaborn           0.12           Estilização de gráficos (heatmap, despine)
#  rich              13.0           Tabelas e painéis coloridos no terminal
#  gc                —              Gerenciamento explícito de memória
#
#  Instalação rápida (Google Colab / pip):
#      !pip install rich pyarrow
#  (numpy, pandas, matplotlib e seaborn já estão disponíveis no Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS GERADA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  TCC_PLD_Project/
#  └── 09-ESCRITA_TCC/
#      └── PARTES_TCC/
#          └── codigos/
#              └── dados/
#                  └── 5_divisão/
#                      ├── treino/
#                      │   └── TREINO_SIN_BRASIL.parquet
#                      ├── teste/
#                      │   └── TESTE_SIN_BRASIL.parquet
#                      └── figuras/
#                          ├── FIG01_proporcao_treino_teste.png
#                          ├── FIG02_timeline_split.png
#                          ├── FIG03_registros_por_ano.png
#                          ├── FIG04_nan_por_ano_split.png
#                          ├── FIG05_heatmap_nan_split.png
#                          ├── FIG06_painel_sintetico.png
#                          └── TAB_AUDITORIA_SPLIT.csv
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Instalar dependências:
#      !pip install rich pyarrow
#
#  Célula 3 — Executar o script:
#      exec(open('05_divisao_treino_teste_v3_ultimos2meses.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import sys
import time
import gc
from pathlib import Path
from contextlib import contextmanager
from typing import Optional, Dict, List, Tuple

def _ensure_pkg(pkg: str, import_as: Optional[str] = None) -> None:
    try:
        __import__(import_as or pkg)
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

for _pkg, _imp in [("rich", None), ("pyarrow", None)]:
    _ensure_pkg(_pkg, _imp)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns

from rich.console import Console
from rich.theme import Theme
from rich.panel import Panel
from rich.table import Table
from rich import box
from rich.rule import Rule

# ==============================================================================
# 🎨 TEMA E CONSOLE
# ==============================================================================

TEMA = Theme({
    "info":    "bold cyan",
    "warning": "bold yellow",
    "error":   "bold red",
    "success": "bold green",
    "header":  "bold blue",
    "label":   "dim white",
    "value":   "bold white",
    "treino":  "bold steel_blue1",
    "teste":   "bold orange1",
    "muted":   "grey62",
})
console = Console(theme=TEMA)

# ==============================================================================
# ⚙️  CONFIGURAÇÕES GLOBAIS
# ==============================================================================

BASE_PATH      = Path("/content/drive/MyDrive/TCC_PLD_Project")
DIR_ENTRADA    = BASE_PATH / "2-DADOS" / "4-DADOS-UNIAO-GERAL"
DIR_SAIDA      = BASE_PATH / "09-ESCRITA_TCC" / "PARTES_TCC" / "codigos" / "dados" / "5_divisão"
DIR_TREINO     = DIR_SAIDA / "treino"
DIR_TESTE      = DIR_SAIDA / "teste"
DIR_FIGURAS    = DIR_SAIDA / "figuras"
ARQUIVO_BRASIL = DIR_ENTRADA / "DADOS_BRASIL_WIDE.parquet"

# 🔧 REGRA DE SPLIT (v3.0): teste = últimos N_MESES_TESTE meses-calendário
# efetivamente presentes no dataset (não mais um ano fixo de corte).
N_MESES_TESTE = 2

KEYS_TIME = ["KEY_ANO", "KEY_MES", "KEY_DIA", "KEY_HORA"]

COR_TREINO = "#2196F3"
COR_TESTE  = "#FF6F00"

REGIOES_ORDEM = ["SUDESTE", "SUL", "NORDESTE", "NORTE"]

TCC_STYLE = {
    "font.family":    "serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi":     120,   # menor que 150 durante geração — salva em 300
    "savefig.dpi":    300,
    "savefig.bbox":   "tight",
}

# ==============================================================================
# ⏱️  UTILITÁRIOS
# ==============================================================================

@contextmanager
def timer(label: str):
    t0 = time.perf_counter()
    yield
    console.print(f"   [muted]⏱  {label}: {time.perf_counter() - t0:.2f}s[/muted]")


def fmt_n(n: int) -> str:
    return f"{n:,}".replace(",", ".")


def fmt_mb(df: pd.DataFrame) -> str:
    """
    FIX [3]: deep=False — NÃO percorre strings/objetos byte a byte.
    Dá uma estimativa rápida suficiente para exibição.
    """
    mb = df.memory_usage(deep=False).sum() / 1024 ** 2
    return f"{mb:.1f} MB"


def fmt_periodo(periodo: int) -> str:
    """Converte um inteiro AAAAMM (ex.: 202511) em 'MM/AAAA' (ex.: '11/2025')."""
    ano, mes = divmod(int(periodo), 100)
    return f"{mes:02d}/{ano}"


def _salvar_fig(fig: plt.Figure, nome: str) -> None:
    DIR_FIGURAS.mkdir(parents=True, exist_ok=True)
    caminho = DIR_FIGURAS / nome
    fig.savefig(caminho, dpi=300, bbox_inches="tight")
    console.print(f"   [success]✔  Figura → {caminho.name}[/success]")


def _otimizar_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """
    FIX [7]: Converte colunas numéricas para o menor dtype possível.
    float64 → float32 reduz memória ~50 % nas colunas de dados.
    Colunas de chave (int) viram int16/int32 conforme o range.
    """
    for col in df.select_dtypes(include="float64").columns:
        df[col] = df[col].astype(np.float32)

    for col in df.select_dtypes(include=["int64", "int32"]).columns:
        col_min = df[col].min()
        col_max = df[col].max()
        if col_min >= np.iinfo(np.int16).min and col_max <= np.iinfo(np.int16).max:
            df[col] = df[col].astype(np.int16)
        elif col_min >= np.iinfo(np.int32).min and col_max <= np.iinfo(np.int32).max:
            df[col] = df[col].astype(np.int32)

    return df


# ==============================================================================
# 🏗️  ENGINE PRINCIPAL
# ==============================================================================

class SplitEngine:
    """
    Divide o dataset Brasil em treino/teste de forma memory-safe, usando uma
    janela móvel de N_MESES_TESTE meses-calendário mais recentes como TESTE.
    Libera df_raw logo após persistir os arquivos Parquet.
    Todas as figuras e tabelas são geradas a partir dos arquivos salvos
    ou de agregações leves — nunca dos DataFrames completos em memória.
    """

    def __init__(self) -> None:
        self.df_raw:    Optional[pd.DataFrame] = None
        self.df_treino: Optional[pd.DataFrame] = None
        self.df_teste:  Optional[pd.DataFrame] = None
        self._meta:     Dict = {}

        # Caminhos dos arquivos persistidos
        self._arq_treino = DIR_TREINO / "TREINO_SIN_BRASIL.parquet"
        self._arq_teste  = DIR_TESTE  / "TESTE_SIN_BRASIL.parquet"

    # ── Pipeline ──────────────────────────────────────────────────────────────

    def run(self) -> None:
        self._print_banner()
        self._criar_dirs()

        with timer("Pipeline total"):
            self._carregar_e_otimizar()
            self._split_e_salvar()        # salva e libera df_raw + subsets
            self._auditar()               # lê de volta só colunas-chave
            self._gerar_figuras()
            self._gerar_tabelas_terminal()

        self._relatorio_final()

    # ── Etapas ────────────────────────────────────────────────────────────────

    def _print_banner(self) -> None:
        console.print()
        console.print(Panel.fit(
            "[header]✂️  SCRIPT 5 — DIVISÃO TREINO / TESTE (v3.0)\n"
            "[white]SIN Brasil │ Split por Janela Móvel │ Memory-Safe[/white]",
            border_style="blue", padding=(1, 4),
        ))
        console.print(
            f"   [label]Regra de corte[/label]: "
            f"[teste]TESTE = últimos {N_MESES_TESTE} meses-calendário do dataset[/teste]  |  "
            f"[treino]TREINO = todo o restante do histórico[/treino]\n"
        )

    def _criar_dirs(self) -> None:
        for d in (DIR_TREINO, DIR_TESTE, DIR_FIGURAS):
            d.mkdir(parents=True, exist_ok=True)
        console.print("   [success]✔  Pastas criadas/verificadas[/success]")

    def _carregar_e_otimizar(self) -> None:
        console.rule("[header]📂 Carregamento e Otimização de Memória[/header]")

        if not ARQUIVO_BRASIL.exists():
            console.print(f"   [error]✖  Não encontrado: {ARQUIVO_BRASIL}[/error]")
            raise FileNotFoundError(str(ARQUIVO_BRASIL))

        with timer("Leitura Parquet"):
            self.df_raw = pd.read_parquet(ARQUIVO_BRASIL)

        r, c = self.df_raw.shape
        console.print(
            f"   [info]Antes da otimização:[/info] "
            f"[value]{fmt_n(r)} linhas × {c} cols[/value]  {fmt_mb(self.df_raw)}"
        )

        # FIX [7]
        with timer("Otimização de dtypes"):
            self.df_raw = _otimizar_dtypes(self.df_raw)
        gc.collect()

        console.print(
            f"   [success]Após otimização:[/success]  {fmt_mb(self.df_raw)}"
        )

        anos = sorted(self.df_raw["KEY_ANO"].unique())
        console.print(
            f"   [label]Anos disponíveis[/label]: "
            f"[value]{anos[0]} → {anos[-1]}[/value]  ({len(anos)} anos)"
        )

        self._meta["n_total"] = r
        self._meta["n_cols"]  = c
        self._meta["anos_disponiveis"] = anos

    def _split_e_salvar(self) -> None:
        """
        FIX [1] + [2] (mantidos) + NOVO [8]:
        - Constrói coluna auxiliar _PERIODO (AAAAMM) só em memória
        - Identifica os N_MESES_TESTE meses-calendário mais recentes
        - Divide por máscara booleana SEM .copy()
        - Salva imediatamente cada subset em Parquet (sem persistir _PERIODO)
        - Libera df_raw, máscaras e coluna auxiliar com del + gc.collect()
        """
        console.rule("[header]✂️  Split e Persistência (Janela Móvel)[/header]")

        # NOVO [8]: período AAAAMM para localizar o corte com precisão mensal
        periodo = (
            self.df_raw["KEY_ANO"].astype(int) * 100
            + self.df_raw["KEY_MES"].astype(int)
        )

        periodos_disponiveis: List[int] = sorted(periodo.unique().tolist())
        if len(periodos_disponiveis) <= N_MESES_TESTE:
            console.print(
                f"   [warning]⚠  Apenas {len(periodos_disponiveis)} meses no dataset "
                f"(≤ N_MESES_TESTE={N_MESES_TESTE}). Split pode ficar desbalanceado.[/warning]"
            )

        periodos_teste = periodos_disponiveis[-N_MESES_TESTE:]
        periodo_corte  = min(periodos_teste)

        meses_teste_fmt = [fmt_periodo(p) for p in periodos_teste]
        console.print(
            f"   [label]Competências de TESTE[/label]: "
            f"[teste]{', '.join(meses_teste_fmt)}[/teste]"
        )
        console.print(
            f"   [label]Corte aplicado[/label]: tudo ≥ [teste]{fmt_periodo(periodo_corte)}[/teste] "
            f"vira TESTE; o restante vira TREINO"
        )

        mask_teste  = periodo >= periodo_corte
        mask_treino = ~mask_teste

        n_total  = self._meta["n_total"]
        n_treino = int(mask_treino.sum())
        n_teste  = int(mask_teste.sum())

        console.print(
            f"   [treino]TREINO[/treino]: {fmt_n(n_treino)} "
            f"({n_treino / n_total * 100:.1f}%)"
        )
        console.print(
            f"   [teste]TESTE[/teste] : {fmt_n(n_teste)} "
            f"({n_teste / n_total * 100:.1f}%)"
        )

        # Salva TREINO
        with timer("Salvar TREINO"):
            (self.df_raw[mask_treino]
                 .to_parquet(self._arq_treino, index=False, compression="snappy"))
        mb_t = self._arq_treino.stat().st_size / 1024 ** 2
        console.print(
            f"   [success]✔  TREINO → {self._arq_treino.name}  ({mb_t:.2f} MB)[/success]"
        )

        # Salva TESTE
        with timer("Salvar TESTE"):
            (self.df_raw[mask_teste]
                 .to_parquet(self._arq_teste, index=False, compression="snappy"))
        mb_te = self._arq_teste.stat().st_size / 1024 ** 2
        console.print(
            f"   [success]✔  TESTE  → {self._arq_teste.name}  ({mb_te:.2f} MB)[/success]"
        )

        # FIX [2]: libera toda a memória dos DataFrames grandes
        del self.df_raw, mask_treino, mask_teste, periodo
        gc.collect()
        console.print("   [muted]🗑  df_raw liberado da memória[/muted]")

        self._meta.update({
            "n_treino": n_treino,
            "n_teste":  n_teste,
            "p_treino": n_treino / n_total * 100,
            "p_teste":  n_teste  / n_total * 100,
            "mb_treino": mb_t,
            "mb_teste":  mb_te,
            "periodos_disponiveis": periodos_disponiveis,
            "periodos_teste":       periodos_teste,
            "periodo_corte":        periodo_corte,
            "meses_teste_fmt":      meses_teste_fmt,
        })

    def _auditar(self) -> None:
        """
        FIX [4]: Lê apenas as colunas necessárias para auditoria — não carrega
        todo o dataset de volta. Usa pyarrow columns filter.
        """
        console.rule("[header]🔍 Auditoria Leve por Ano[/header]")

        def _auditar_arquivo(caminho: Path, label: str) -> pd.DataFrame:
            # Lê só KEY_ANO, KEY_MES + colunas de dados em float32
            df_keys = pd.read_parquet(caminho, columns=KEYS_TIME)
            # Para calcular NaN, precisa das colunas de dados
            # Lê separado para controle de memória
            df_dados = pd.read_parquet(caminho)
            cols_dados = [c for c in df_dados.columns if c not in KEYS_TIME]

            registros = []
            for ano, grp in df_dados.groupby("KEY_ANO"):
                pct_nan = grp[cols_dados].isna().mean().mean() * 100
                meses   = grp["KEY_MES"].nunique()
                registros.append({
                    "Conjunto":       label,
                    "Ano":            int(ano),
                    "Registros":      len(grp),
                    "Meses c/ Dados": meses,
                    "NaN (%)":        round(pct_nan, 2),
                    "Completude (%)": round(100 - pct_nan, 2),
                })

            del df_dados, df_keys
            gc.collect()
            return pd.DataFrame(registros)

        with timer("Auditoria TREINO"):
            audit_t = _auditar_arquivo(self._arq_treino, "TREINO")

        with timer("Auditoria TESTE"):
            audit_te = _auditar_arquivo(self._arq_teste, "TESTE")

        audit_total = pd.concat([audit_t, audit_te], ignore_index=True)

        # Salva CSV de auditoria
        arq_audit = DIR_FIGURAS / "TAB_AUDITORIA_SPLIT.csv"
        audit_total.to_csv(arq_audit, index=False, encoding="utf-8-sig")
        console.print(f"   [success]✔  Auditoria CSV → {arq_audit.name}[/success]")

        self._meta["audit_treino"] = audit_t
        self._meta["audit_teste"]  = audit_te
        self._meta["audit_total"]  = audit_total

        console.print("   [success]✔  Auditoria concluída[/success]")

    # ── Figuras ───────────────────────────────────────────────────────────────

    def _gerar_figuras(self) -> None:
        console.rule("[header]📊 Geração de Figuras TCC[/header]")
        # FIX [5]: try/finally em cada figura garante plt.close() mesmo com erro
        self._fig_pizza_split()
        self._fig_timeline_split()
        self._fig_registros_por_ano()
        self._fig_nan_por_ano()
        self._fig_heatmap_nan()
        self._fig_painel_geral()
        gc.collect()

    def _fig_pizza_split(self) -> None:
        console.print("   [info]↳ FIG01 — Proporção Treino/Teste[/info]")
        m = self._meta
        fig = None
        try:
            with plt.rc_context(TCC_STYLE):
                fig, ax = plt.subplots(figsize=(6, 5))
                vals   = [m["n_treino"], m["n_teste"]]
                cores  = [COR_TREINO, COR_TESTE]
                wedges, _, autotexts = ax.pie(
                    vals, colors=cores, autopct="%1.1f%%",
                    startangle=90, pctdistance=0.72,
                    wedgeprops={"linewidth": 2, "edgecolor": "white", "width": 0.50},
                )
                for at in autotexts:
                    at.set_fontsize(11); at.set_fontweight("bold"); at.set_color("white")

                ax.text(0,  0.12, fmt_n(m["n_total"]), ha="center",
                        fontsize=15, fontweight="bold")
                ax.text(0, -0.16, "registros\nno total", ha="center",
                        fontsize=9, color="gray", linespacing=1.4)

                meses_teste_txt = ", ".join(m["meses_teste_fmt"])
                patches = [
                    mpatches.Patch(facecolor=COR_TREINO,
                                   label=f"Treino (histórico) — {m['p_treino']:.1f}%"),
                    mpatches.Patch(facecolor=COR_TESTE,
                                   label=f"Teste ({meses_teste_txt}) — {m['p_teste']:.1f}%"),
                ]
                ax.legend(handles=patches, loc="lower center",
                          bbox_to_anchor=(0.5, -0.16), ncol=1, framealpha=0.85, fontsize=8.5)
                ax.set_title(
                    f"Figura 1 — Proporção Treino/Teste "
                    f"(Teste = últimos {N_MESES_TESTE} meses do dataset)",
                    fontweight="bold", pad=16,
                )
                plt.tight_layout()
            _salvar_fig(fig, "FIG01_proporcao_treino_teste.png")
        finally:
            if fig:
                plt.close(fig)

    def _fig_timeline_split(self) -> None:
        """
        NOVO [9]: granularidade MENSAL (não mais anual), pois o corte agora
        pode cair no meio de um ano. Rótulo de ano aparece apenas em Janeiro
        de cada ano, para não poluir o eixo.
        """
        console.print("   [info]↳ FIG02 — Timeline Mensal com Ponto de Corte[/info]")
        periodos = self._meta["periodos_disponiveis"]
        periodo_corte = self._meta["periodo_corte"]
        fig = None
        try:
            with plt.rc_context(TCC_STYLE):
                fig, ax = plt.subplots(figsize=(16, 3))
                for idx, p in enumerate(periodos):
                    ano, mes = divmod(p, 100)
                    cor = COR_TESTE if p >= periodo_corte else COR_TREINO
                    ax.barh(0, 1, left=idx - 0.5, height=0.55,
                            color=cor, edgecolor="white", linewidth=0.8)
                    rotulo = f"{ano}" if mes == 1 else ""
                    if rotulo:
                        ax.text(idx, 0, rotulo, ha="center", va="center",
                                fontsize=8, fontweight="bold", color="white", rotation=90)

                idx_corte = periodos.index(periodo_corte)
                ax.axvline(idx_corte - 0.5, color="#333",
                           linewidth=2, linestyle="--", zorder=5)
                ax.text(idx_corte - 0.4, 0.35,
                        f"  Corte: {fmt_periodo(periodo_corte)}",
                        fontsize=9, color="#333", va="center", fontweight="bold")

                patches = [
                    mpatches.Patch(facecolor=COR_TREINO, label="Treino (histórico)"),
                    mpatches.Patch(facecolor=COR_TESTE,
                                   label=f"Teste (últimos {N_MESES_TESTE} meses)"),
                ]
                ax.legend(handles=patches, loc="upper left", framealpha=0.85, fontsize=9)
                ax.set_xlim(-0.6, len(periodos) - 0.4)
                ax.set_ylim(-0.5, 0.65)
                ax.set_yticks([]); ax.set_xticks([])
                ax.set_title(
                    "Figura 2 — Linha do Tempo Mensal: Divisão Treino / Teste (SIN Brasil)",
                    fontweight="bold",
                )
                sns.despine(left=True, bottom=True, ax=ax)
                plt.tight_layout()
            _salvar_fig(fig, "FIG02_timeline_split.png")
        finally:
            if fig:
                plt.close(fig)

    def _fig_registros_por_ano(self) -> None:
        """
        Ajustado para o novo split: como o corte pode cair no meio de um ano,
        cada ano exibe barras empilhadas (Treino + Teste) quando aplicável —
        os dados já vêm corretamente segregados por mês em `audit_total`.
        """
        console.print("   [info]↳ FIG03 — Registros por Ano (Treino/Teste empilhado)[/info]")
        m = self._meta
        audit = m["audit_total"][["Ano", "Conjunto", "Registros"]].copy()
        pivot = audit.pivot_table(index="Ano", columns="Conjunto",
                                   values="Registros", aggfunc="sum").fillna(0)
        for col in ["TREINO", "TESTE"]:
            if col not in pivot.columns:
                pivot[col] = 0
        pivot = pivot[["TREINO", "TESTE"]].sort_index()

        fig = None
        try:
            with plt.rc_context(TCC_STYLE):
                fig, ax = plt.subplots(figsize=(11, 5))
                anos_str = [str(int(a)) for a in pivot.index]
                base = np.zeros(len(pivot))

                for col, cor in [("TREINO", COR_TREINO), ("TESTE", COR_TESTE)]:
                    vals = pivot[col].values
                    ax.bar(anos_str, vals, bottom=base, color=cor,
                           edgecolor="white", linewidth=0.7, alpha=0.90, label=col.title())
                    for x, (v, b) in enumerate(zip(vals, base)):
                        if v > 0:
                            ax.text(x, b + v / 2, fmt_n(int(v)), ha="center", va="center",
                                    fontsize=7.5, fontweight="bold", color="white")
                    base = base + vals

                patches = [
                    mpatches.Patch(facecolor=COR_TREINO, label="Treino"),
                    mpatches.Patch(facecolor=COR_TESTE,  label=f"Teste (últimos {N_MESES_TESTE} meses)"),
                ]
                ax.legend(handles=patches, framealpha=0.85)
                ax.set_title(
                    "Figura 3 — Registros por Ano com Identificação Treino/Teste (SIN Brasil)",
                    fontweight="bold",
                )
                ax.set_xlabel("Ano"); ax.set_ylabel("Nº de Registros")
                ax.yaxis.set_major_formatter(
                    mticker.FuncFormatter(lambda x, _: f"{int(x):,}".replace(",", "."))
                )
                ax.grid(axis="y", linestyle="--", alpha=0.35)
                sns.despine(ax=ax)
                plt.tight_layout()
            _salvar_fig(fig, "FIG03_registros_por_ano.png")
        finally:
            if fig:
                plt.close(fig)

    def _fig_nan_por_ano(self) -> None:
        console.print("   [info]↳ FIG04 — Dados Faltantes por Ano[/info]")
        audit = self._meta["audit_total"]
        fig = None
        try:
            with plt.rc_context(TCC_STYLE):
                fig, ax = plt.subplots(figsize=(11, 4.5))
                # Como um mesmo ano pode ter barras de Treino e Teste (corte no
                # meio do ano), usamos deslocamento lado a lado em vez de sobrepor.
                anos = sorted(audit["Ano"].unique())
                largura = 0.38
                for i, conjunto in enumerate(["TREINO", "TESTE"]):
                    sub = audit[audit["Conjunto"] == conjunto].set_index("Ano")
                    cor = COR_TREINO if conjunto == "TREINO" else COR_TESTE
                    offset = (-largura / 2) if i == 0 else (largura / 2)
                    xs = [anos.index(a) + offset for a in sub.index]
                    vals = sub["NaN (%)"].values
                    ax.bar(xs, vals, width=largura, color=cor,
                           edgecolor="white", linewidth=0.7, alpha=0.88)
                    for x, v in zip(xs, vals):
                        ax.text(x, v + 0.3, f"{v:.1f}%", ha="center", va="bottom",
                                fontsize=7, fontweight="bold")

                ax.set_xticks(range(len(anos)))
                ax.set_xticklabels([str(int(a)) for a in anos])

                patches = [
                    mpatches.Patch(facecolor=COR_TREINO, label="Treino"),
                    mpatches.Patch(facecolor=COR_TESTE,  label=f"Teste (últimos {N_MESES_TESTE} meses)"),
                ]
                ax.legend(handles=patches, framealpha=0.85)
                ax.set_title(
                    "Figura 4 — Percentual Médio de Dados Faltantes por Ano (Treino vs Teste)",
                    fontweight="bold",
                )
                ax.set_xlabel("Ano"); ax.set_ylabel("% NaN Médio")
                ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))
                ax.grid(axis="y", linestyle="--", alpha=0.35)
                sns.despine(ax=ax)
                plt.tight_layout()
            _salvar_fig(fig, "FIG04_nan_por_ano_split.png")
        finally:
            if fig:
                plt.close(fig)

    def _fig_heatmap_nan(self) -> None:
        """
        FIX [6]: Lê o arquivo consolidado em chunks, calculando NaN por região
        sem manter o dataset inteiro na memória.
        """
        console.print("   [info]↳ FIG05 — Heatmap NaN Região × Ano[/info]")

        # Descobre colunas por região no arquivo treino (sem carregar tudo)
        schema = pd.read_parquet(self._arq_treino, columns=KEYS_TIME[:1]).columns
        all_cols = pd.read_parquet(self._arq_treino).columns.tolist()
        cols_por_regiao = {
            r: [c for c in all_cols if c.endswith(f"_{r}")]
            for r in REGIOES_ORDEM
        }
        del all_cols
        gc.collect()

        registros = []
        for arq, label in [(self._arq_treino, "TREINO"), (self._arq_teste, "TESTE")]:
            # Lê somente chave de ano + colunas de região
            for regiao, cols_reg in cols_por_regiao.items():
                if not cols_reg:
                    continue
                df_r = pd.read_parquet(arq, columns=["KEY_ANO"] + cols_reg)
                for ano, grp in df_r.groupby("KEY_ANO"):
                    pct = grp[cols_reg].isna().mean().mean() * 100
                    registros.append({
                        "Região": regiao,
                        "Ano":    int(ano),
                        "NaN (%)": round(pct, 2),
                    })
                del df_r
                gc.collect()

        if not registros:
            console.print("   [warning]⚠  Sem colunas por região para Figura 5.[/warning]")
            return

        pivot = (pd.DataFrame(registros)
                   .pivot(index="Região", columns="Ano", values="NaN (%)"))

        fig = None
        try:
            with plt.rc_context(TCC_STYLE):
                fig, ax = plt.subplots(figsize=(13, 4))
                sns.heatmap(
                    pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
                    linewidths=0.4, linecolor="white",
                    vmin=0, vmax=100, ax=ax,
                    cbar_kws={"label": "% NaN", "shrink": 0.8},
                )
                ax.set_title(
                    "Figura 5 — Completude dos Dados por Região e Ano (% NaN)",
                    fontweight="bold",
                )
                ax.set_xlabel("Ano"); ax.set_ylabel("")
                plt.tight_layout()
            _salvar_fig(fig, "FIG05_heatmap_nan_split.png")
        finally:
            if fig:
                plt.close(fig)

    def _fig_painel_geral(self) -> None:
        console.print("   [info]↳ FIG06 — Painel Sintético 2×2[/info]")
        m = self._meta
        audit_t  = m["audit_treino"]
        audit_te = m["audit_teste"]
        periodos = m["periodos_disponiveis"]
        periodo_corte = m["periodo_corte"]
        audit_all = m["audit_total"]

        fig = None
        try:
            with plt.rc_context(TCC_STYLE):
                fig = plt.figure(figsize=(14, 10))
                gs  = gridspec.GridSpec(2, 2, hspace=0.40, wspace=0.32)
                ax_pizza    = fig.add_subplot(gs[0, 0])
                ax_timeline = fig.add_subplot(gs[0, 1])
                ax_bars     = fig.add_subplot(gs[1, 0])
                ax_mensal   = fig.add_subplot(gs[1, 1])

                # Pizza
                vals  = [m["n_treino"], m["n_teste"]]
                cores = [COR_TREINO, COR_TESTE]
                _, _, autotexts = ax_pizza.pie(
                    vals, colors=cores, autopct="%1.1f%%",
                    startangle=90, pctdistance=0.72,
                    wedgeprops={"linewidth": 1.8, "edgecolor": "white", "width": 0.48},
                )
                for at in autotexts:
                    at.set_fontsize(9.5); at.set_fontweight("bold"); at.set_color("white")
                ax_pizza.text(0, 0.10, fmt_n(m["n_total"]), ha="center",
                              fontsize=13, fontweight="bold")
                ax_pizza.text(0, -0.17, "total", ha="center", fontsize=9, color="gray")
                ax_pizza.set_title("Proporção Treino/Teste", fontweight="bold", fontsize=11)
                ax_pizza.legend(handles=[
                    mpatches.Patch(color=COR_TREINO, label=f"Treino {m['p_treino']:.1f}%"),
                    mpatches.Patch(color=COR_TESTE,  label=f"Teste {m['p_teste']:.1f}%"),
                ], loc="lower center", bbox_to_anchor=(0.5, -0.15),
                   ncol=2, fontsize=8.5, framealpha=0.85)

                # Timeline mensal (NOVO [9])
                for idx, p in enumerate(periodos):
                    ano, mes = divmod(p, 100)
                    cor = COR_TESTE if p >= periodo_corte else COR_TREINO
                    ax_timeline.barh(0, 1, left=idx - 0.5, height=0.50,
                                     color=cor, edgecolor="white", linewidth=0.6)
                idx_corte = periodos.index(periodo_corte)
                ax_timeline.axvline(idx_corte - 0.5, color="#333",
                                    linewidth=2, linestyle="--")
                ax_timeline.set_xlim(-0.6, len(periodos) - 0.4)
                ax_timeline.set_ylim(-0.45, 0.6)
                ax_timeline.set_yticks([]); ax_timeline.set_xticks([])
                ax_timeline.set_title("Linha do Tempo Mensal Treino/Teste", fontweight="bold", fontsize=11)
                sns.despine(left=True, bottom=True, ax=ax_timeline)

                # Barras por ano (empilhado Treino/Teste)
                pivot_bar = audit_all.pivot_table(
                    index="Ano", columns="Conjunto", values="Registros", aggfunc="sum"
                ).fillna(0)
                for col in ["TREINO", "TESTE"]:
                    if col not in pivot_bar.columns:
                        pivot_bar[col] = 0
                pivot_bar = pivot_bar[["TREINO", "TESTE"]].sort_index()
                anos_str = [str(int(a)) for a in pivot_bar.index]
                base = np.zeros(len(pivot_bar))
                for col, cor in [("TREINO", COR_TREINO), ("TESTE", COR_TESTE)]:
                    vals = pivot_bar[col].values
                    ax_bars.bar(anos_str, vals, bottom=base, color=cor,
                                edgecolor="white", linewidth=0.6, alpha=0.90)
                    base = base + vals
                ax_bars.yaxis.set_major_formatter(
                    mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}k")
                )
                ax_bars.set_title("Registros por Ano (empilhado)", fontweight="bold", fontsize=11)
                ax_bars.set_xlabel("Ano"); ax_bars.set_ylabel("Registros")
                ax_bars.tick_params(axis="x", rotation=45)
                ax_bars.grid(axis="y", linestyle="--", alpha=0.30)
                sns.despine(ax=ax_bars)

                # Completude por ano
                comp_treino = audit_t.set_index("Ano")["Completude (%)"]
                comp_teste  = audit_te.set_index("Ano")["Completude (%)"]
                ax_mensal.plot(comp_treino.index.astype(str), comp_treino.values,
                               color=COR_TREINO, marker="o", linewidth=2,
                               markersize=6, label="Treino")
                ax_mensal.plot(comp_teste.index.astype(str), comp_teste.values,
                               color=COR_TESTE, marker="s", linewidth=2,
                               markersize=6, label="Teste")
                ax_mensal.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
                ax_mensal.set_ylim(0, 110)
                ax_mensal.set_title("Completude Média por Ano (%)", fontweight="bold", fontsize=11)
                ax_mensal.set_xlabel("Ano"); ax_mensal.set_ylabel("Completude (%)")
                ax_mensal.tick_params(axis="x", rotation=45)
                ax_mensal.legend(framealpha=0.85)
                ax_mensal.grid(linestyle="--", alpha=0.30)
                sns.despine(ax=ax_mensal)

                fig.suptitle(
                    f"Figura 6 — Painel Sintético de Divisão Treino/Teste "
                    f"(SIN Brasil — Teste = últimos {N_MESES_TESTE} meses: "
                    f"{', '.join(m['meses_teste_fmt'])})",
                    fontweight="bold", y=1.01, fontsize=13,
                )
            _salvar_fig(fig, "FIG06_painel_sintetico.png")
        finally:
            if fig:
                plt.close(fig)

    # ── Tabelas no terminal ───────────────────────────────────────────────────

    def _gerar_tabelas_terminal(self) -> None:
        console.rule("[header]📋 Relatórios Tabulares[/header]")
        self._tabela_resumo_split()
        self._tabela_auditoria_anual()
        self._tabela_cobertura_mensal()

    def _tabela_resumo_split(self) -> None:
        m = self._meta
        t = Table(
            title="Tabela 1 — Resumo Geral do Split Treino/Teste",
            box=box.DOUBLE_EDGE, border_style="blue",
            header_style="bold white on blue",
        )
        t.add_column("Conjunto",     style="bold", justify="center")
        t.add_column("Competências", justify="center")
        t.add_column("Registros",    justify="right")
        t.add_column("% do Total",   justify="right")
        t.add_column("Colunas",      justify="center")
        t.add_column("Arquivo",      justify="right")

        t.add_row(
            "[treino]TREINO[/treino]", "histórico completo (exceto teste)",
            f"[treino]{fmt_n(m['n_treino'])}[/treino]",
            f"[treino]{m['p_treino']:.1f}%[/treino]",
            str(m["n_cols"]), f"{m['mb_treino']:.2f} MB",
        )
        t.add_row(
            "[teste]TESTE[/teste]", ", ".join(m["meses_teste_fmt"]),
            f"[teste]{fmt_n(m['n_teste'])}[/teste]",
            f"[teste]{m['p_teste']:.1f}%[/teste]",
            str(m["n_cols"]), f"{m['mb_teste']:.2f} MB",
        )
        t.add_row(
            "[bold white]TOTAL[/bold white]", "—",
            f"[bold]{fmt_n(m['n_total'])}[/bold]",
            "100.0%", str(m["n_cols"]), "—", style="dim",
        )
        console.print(t)

    def _tabela_auditoria_anual(self) -> None:
        audit = self._meta["audit_total"]
        t = Table(
            title="Tabela 2 — Auditoria Temporal por Ano",
            box=box.SIMPLE_HEAD, border_style="blue",
            header_style="bold white on blue",
        )
        for col in ["Conjunto", "Ano", "Registros", "Meses c/ Dados", "NaN (%)", "Completude (%)"]:
            t.add_column(col, justify="center" if col in ("Conjunto", "Ano") else "right")

        for _, row in audit.iterrows():
            cor = "treino" if row["Conjunto"] == "TREINO" else "teste"
            nan_cor = "green" if row["NaN (%)"] < 10 else ("yellow" if row["NaN (%)"] < 40 else "red")
            t.add_row(
                f"[{cor}]{row['Conjunto']}[/{cor}]",
                str(int(row["Ano"])),
                fmt_n(int(row["Registros"])),
                str(int(row["Meses c/ Dados"])),
                f"[{nan_cor}]{row['NaN (%)']:.2f}%[/{nan_cor}]",
                f"{row['Completude (%)']:.2f}%",
            )
        console.print(t)

    def _tabela_cobertura_mensal(self) -> None:
        """Lê apenas KEY_MES dos arquivos já salvos — evita recarregar tudo."""
        mensal_t  = pd.read_parquet(self._arq_treino, columns=["KEY_MES"])["KEY_MES"].value_counts()
        mensal_te = pd.read_parquet(self._arq_teste,  columns=["KEY_MES"])["KEY_MES"].value_counts() \
                    if self._arq_teste.exists() else pd.Series(dtype=int)

        meses = ["Jan","Fev","Mar","Abr","Mai","Jun",
                 "Jul","Ago","Set","Out","Nov","Dez"]

        t = Table(
            title="Tabela 3 — Distribuição Mensal de Registros (Treino vs Teste)",
            box=box.SIMPLE_HEAD, border_style="blue",
            header_style="bold white on blue",
        )
        t.add_column("Mês",    justify="center")
        t.add_column("Treino", justify="right")
        t.add_column("Teste",  justify="right")
        t.add_column("Total",  justify="right")

        for i in range(1, 13):
            val_t  = int(mensal_t.get(i, 0))
            val_te = int(mensal_te.get(i, 0))
            t.add_row(
                meses[i - 1],
                f"[treino]{fmt_n(val_t)}[/treino]"  if val_t  else "[muted]—[/muted]",
                f"[teste]{fmt_n(val_te)}[/teste]"   if val_te else "[muted]—[/muted]",
                fmt_n(val_t + val_te),
            )
        console.print(t)

    # ── Relatório final ───────────────────────────────────────────────────────

    def _relatorio_final(self) -> None:
        m = self._meta
        n_figs = len(list(DIR_FIGURAS.glob("*.png")))
        console.print()
        console.print(Panel(
            f"[success]✅ PIPELINE CONCLUÍDO COM SUCESSO[/success]\n\n"
            f"   [label]Treino[/label]: [treino]{fmt_n(m['n_treino'])} registros[/treino] "
            f"({m['p_treino']:.1f}%)  →  {self._arq_treino.name}  ({m['mb_treino']:.2f} MB)\n"
            f"   [label]Teste[/label] : [teste]{fmt_n(m['n_teste'])} registros[/teste] "
            f"({m['p_teste']:.1f}%)  →  {self._arq_teste.name}  ({m['mb_teste']:.2f} MB)\n"
            f"   [label]Competências de Teste[/label]: {', '.join(m['meses_teste_fmt'])}\n\n"
            f"   [label]Figuras geradas[/label]: {n_figs}\n"
            f"   [label]Pasta de saída[/label]  : {DIR_SAIDA}",
            title="[header]RELATÓRIO FINAL — SCRIPT 05 (v3.0)[/header]",
            border_style="green", padding=(1, 4),
        ))


# ==============================================================================
# 🚀 ENTRYPOINT
# ==============================================================================

if __name__ == "__main__":
    engine = SplitEngine()
    engine.run()